# ***Industrial Vision SLM Benchmark***

## ***Notebook 00 — Préparation de l'environnement***

**Objectif :** vérifier l'environnement Python, la disponibilité
du GPU NVIDIA T4 et les dépendances nécessaires au projet.


## **1. Vérification de l'environnement Python et GPU**

In [ ]:
import sys
import torch

print("Python :", sys.version.split()[0])
print("PyTorch :", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

    props = torch.cuda.get_device_properties(0)
    print(f"VRAM totale : {props.total_memory / 1024**3} Gio")
else:
    print("Aucun GPU CUDA détecté.")

## **2. Test de calcul matriciel sur GPU**

In [ ]:
import torch

# 1. Sélection du GPU CUDA
device = torch.device("cuda")

# 2. Création de deux matrices de 1024x1024 directement dans le GPU
A = torch.randn(1024, 1024, device=device)
B = torch.randn(1024, 1024, device=device)

# 3. Multiplication matricielle sur le GPU
C = A @ B

# 4. Attente de la fin des calculs
torch.cuda.synchronize()

# 5. Vérif du périphérique et du résultat
print("GPU utilisé :", C.device)
print("DImensions du résultat :", C.shape)
print("Résultat valide :",torch.isfinite(C).all().item())

# 6. Mesure de la mémoire GPU occupée par les tenseurs PyTorch
memory = torch.cuda.memory_allocated() / 1024**2
print(f"Memoire GPU allouée : {memory:.2f} Mio")

# 7. Libération des mtrices devenues inutiles
del A, B, C
torch.cuda.empty_cache()

## **3. Vérification des bibliothèques nécessaires**

### *3.1. Inventaire des dépendances Python*

In [ ]:
# ==================================================================
# Section 3.1 - Inventaire des bibliothèques Python
# ==================================================================

from importlib.metadata import version, PackageNotFoundError

# Bibliothèques nécessaires au projet
packages = [
    "torch", "transformers", "accelerate", "peft",
    "bitsandbytes", "datasets", "sentence-transformers",
    "faiss-cpu", "pypdf", "trl", "streamlit",
]

print("=== Vérification des dépendances ===\n")

for package in packages:
    try:
        # Récuperer la version installée
        installed_version = version(package)
        print(f"[OK] {package:<24} {installed_version}")
    except PackageNotFoundError:
        print(f"[--] {package:<24} Non installé")

print("\nVérification terminée.")

## **4. Installation des dépendances compatibles**

### *4.1. Installation des bibliothèques manquantes*

In [ ]:
# ========================================================
# Section 4.1 - Installation des dépendances manquantes
# ========================================================
%pip install bitsandbytes faiss-cpu pypdf

### *4.2. Vérification des dépendances installées*

In [ ]:
# =========================================================
# Section 4.2 - Vérif des bibliothèques installées
# =========================================================
from importlib.metadata import version
import torch
import faiss
import bitsandbytes
import pypdf

# Affichage des versions des bibliothèques
packages = [
    "torch", "bitsandbytes", "faiss-cpu", "pypdf",
]

print("=== Versions installées ===\n")

for package in packages:
    print(f"{package:<16} : {version(package)}")

# Vérifier que PyTorch détacte toujours notre GPU
print("\n=== Vérification GPU ===\n")

print("CUDA disponible :", torch.cuda.is_available())
print("Version CUDA PyTorch :", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

print("\nVérification terminée.")

## **5. Connexion à Google Drive**

### *5.1. Montage et vérification de Google Drive*

In [ ]:
# ============================================================
# Section 5.1 - Montage de Google Drive
# ============================================================
from google.colab import drive
from pathlib import Path

# 1. Connexion Google Drive à notre environnement Colab
drive.mount("/content/drive")

# 2. Définnition du chemin racine de notre espace Drive
drive_root = Path("/content/drive/MyDrive")

# 3. Vérif que le répertoire est accessible
print("\n=== Vérification Google Drive ===\n")

print("Chemin :", drive_root)
print("Drive accessible :", drive_root.is_dir())

if drive_root.is_dir():
    print("[OK] Google Drive est correctement monté.")
else:
    print("[ERREUR] Google Drive est inacessible.")


## **6. Préparation de la structure du projet**

### *6.1. Création des répertoires de stockage*

In [ ]:
# =======================================================
# Section 6.1 - Création des répertoires du projet
# =======================================================
from pathlib import Path

# 1. Définition du répertoire principal du projet sur Drive
PROJECT_ROOT = Path("/content/drive/MyDrive/industrial-vision-slm-benchmark")

# 2. Définir les sous-répertoires nécessaires
directories = [
    "data/raw", "data/processed", "data/instructions", "data/preferences", "data/evaluations",
    "outputs/vector_index", "outputs/checkpoints", "outputs/adapters", "outputs/metrics", "outputs/figures",
]

# 3. Création des répertoires
for directory in directories:
    path = PROJECT_ROOT / directory
    path.mkdir(parents=True, exist_ok=True)

    print(f"[OK] {directory}")

print("\nRépertoire principal :", PROJECT_ROOT)
print("Nombre de sous-répertoires :", len(directories))

### *6.2. Vérification de l'arborescence*

In [ ]:
# ============================================================
# Section 6.2 - Vérification de l'arborescence
# ============================================================
missing = []

for directory in directories:
    path = PROJECT_ROOT / directory
    if not path.is_dir():
        missing.append(directory)

if not missing:
    print("[OK] Tous les répertoires existent.")
    print(f"[OK] {len(directories)} sous-répertoires vérifiés.")
else:
    print("[ERREUR] Répertoires manquants :")
    for directory in missing:
        print(" -", directory)